In [1]:
import os
from pathlib import Path

DATA_DIR = Path("data")

def load_documents(data_dir=DATA_DIR):
    docs = []
    for path in data_dir.glob("*.txt"):
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        docs.append({"filename": path.name, "text": text})
    return docs

docs = load_documents()
len(docs), docs[0]["filename"]


(1, 'sharelink.txt')

In [2]:
def chunk_text(text, max_chars=800):
    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chars
        chunk = text[start:end]
        chunks.append(chunk)
        start = end
    return chunks

chunks = []
for doc in docs:
    for chunk in chunk_text(doc["text"]):
        chunks.append({
            "source": doc["filename"],
            "text": chunk
        })

len(chunks), chunks[0]

(15,
 {'source': 'sharelink.txt',
  'text': 'Introduction:\nAbout the ShareLink Pro 1000 The ShareLink Pro 1000 Wireless and Wired Collaboration Gateway enables anyone to present wireless or wired content from their computers, tablets, or smartphones onto a display for easy collaboration. It features streaming technology that supports simultaneous display of up to four content sources, including an HDMI-connected device. The HDMI input supports wired connections from any connected source in the room. To support a wide range of environments, the  ShareLink Pro 1000 has collaboration and moderator modes that facilitate both open and restrictive environments. The ShareLink Pro 1000 provides easy integration of AV and mobile devices into meeting, huddle, collaboration, and presentation spaces.\nFeatures:\nWirelessly share content from mobil'})

In [ ]:
%pip install sentence_transformers

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")  # small, fast model

chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = model.encode(chunk_texts, convert_to_numpy=True)
chunk_embeddings.shape

d:\MachineLearningCourse\Projects\extron_qa_proj\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2672.32it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(15, 384)

In [ ]:
def retrieve_relevant_chunks(query, top_k=5):
    query_emb = model.encode([query], convert_to_numpy=True)
    sims = cosine_similarity(query_emb, chunk_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    results = []
    for idx in top_idx:
        results.append({
            "score": float(sims[idx]),
            "source": chunks[idx]["source"],
            "text": chunks[idx]["text"]
        })
    return results

results = retrieve_relevant_chunks("What is the sharelink used for?")
results[0]

{'score': 0.20610502362251282,
 'source': 'sharelink.txt',
 'text': 'ce. • WebView technology displays slide images on attendee’s personal devices via a Web browser — The ShareLink Pro 1000 enables meeting content to display on a participant’s mobile device. This is ideal for attendees who cannot easily view the main display. • 128-bit data encryption — A variety of security protocols ensure that all content transmitted between devices and the ShareLink Pro 1000 is fully encrypted and secure. • Power over Ethernet (PoE+) allows the ShareLink Pro to receive power and communication over a single Ethernet cable, eliminating the need for a local power supply  Dual Gigabit Ethernet — Provides two high-speed data links, enabling segmentation of guest and private networks for fast and easy access to the web or other network resources. • Fully customizable welcome '}

In [ ]:
def answer_question(query, top_k=3):
    results = retrieve_relevant_chunks(query, top_k=top_k)
    print(f"Question: {query}\n")
    print("Most relevant information:\n")
    for r in results:
        print(f"Source: {r['source']} (score: {r['score']:.3f})")
        print(r["text"])
        print("-" * 80)

answer_question("What are the main features of the sharelink?")

In [ ]:
def build_context(results):
    context = ""
    for r in results:
        context += f"From {r['source']}:\n{r['text']}\n\n"
    return context

def build_prompt(query, results):
    context = build_context(results)
    prompt = f"""You are an assistant answering questions about Extron products.

Use ONLY the information in the context below. If the answer is not there, say you don't know.

Context:
{context}

Question: {query}
Answer:"""
    return prompt

### Fully Local Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

llm_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(llm_name)
model_llm = AutoModelForCausalLM.from_pretrained(llm_name)

In [ ]:
def build_prompt(query, retrieved_chunks):
    context = ""
    for r in retrieved_chunks:
        context += f"Source: {r['source']}\n{r['text']}\n\n"

    prompt = f"""
You are a helpful assistant answering questions about Extron products.

Use ONLY the information in the context below. If the answer is not in the context, say you don't know.

Context:
{context}

Question: {query}

Answer:
"""
    return prompt

In [ ]:
import torch

def generate_answer(prompt, max_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model_llm.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer[len(prompt):].strip()

In [ ]:
def ask_extron(query, top_k=3):
    retrieved = retrieve_relevant_chunks(query, top_k=top_k)
    prompt = build_prompt(query, retrieved)
    answer = generate_answer(prompt)
    
    print("QUESTION:")
    print(query)
    print("\nANSWER:")
    print(answer)
    
    print("\n--- Retrieved Chunks Used ---")
    for r in retrieved:
        print(f"{r['source']} (score {r['score']:.3f})")

In [ ]:
ask_extron("What are the main features of the sharelink?")

### Chat History

In [ ]:
chat_history = []

In [ ]:
def format_chat_history(history):
    text = ""
    for turn in history:
        text += f"User: {turn['user']}\n"
        text += f"Assistant: {turn['assistant']}\n\n"
    return text

In [ ]:
def build_chat_prompt(query, retrieved_chunks, history):
    context = ""
    for r in retrieved_chunks:
        context += f"Source: {r['source']}\n{r['text']}\n\n"

    history_text = format_chat_history(history)

    prompt = f"""
You are a helpful assistant answering questions about Extron products.

Use ONLY the information in the context below. If the answer is not in the context, say you don't know.

Conversation so far:
{history_text}

Context:
{context}

User: {query}
Assistant:
"""
    return prompt

In [ ]:
import torch

def generate_answer(prompt, max_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model_llm.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    # Decode only the newly generated text
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = full_output[len(prompt):].strip()
    return answer

In [ ]:
def chat(query, top_k=3):
    # Retrieve relevant chunks
    retrieved = retrieve_relevant_chunks(query, top_k=top_k)

    # Build prompt with history
    prompt = build_chat_prompt(query, retrieved, chat_history)

    # Generate answer
    answer = generate_answer(prompt)

    # Save to history
    chat_history.append({
        "user": query,
        "assistant": answer
    })

    # Display
    print(f"User: {query}\n")
    print(f"Assistant: {answer}\n")
    print("--- Retrieved Chunks Used ---")
    for r in retrieved:
        print(f"{r['source']} (score {r['score']:.3f})")

In [ ]:
chat("What does the sharelink do?")